# ML-04 — Search Intelligence Data Contract

**Lane:** Refresh / Content Opportunity Scoring

This notebook defines and verifies the warehouse slice used for the lane. The development month is **March 2026**, a mid-panel month rather than the final `_sample` month.

## 1. Unit of analysis + time window

**One row = one daily content-performance observation for one client and one content item.**

For this exercise I use the March 2026 partition. I use a mid-panel month because the warehouse guidance says the final `_sample` is June 2026 and should be treated as a sealed outcome/test month.

## 2. Fields: feature / label / context / excluded

**Features:** pre-decision performance/search fields that are available in the chosen snapshot.

**Label / proxy:** a future decline outcome, when a past→future label is defined. It is never used as a feature.

**Context:** `client_id`, `content_id`, and dates used for grouping, joining, filtering, or splitting. IDs are not model features.

**Excluded:** `trend_direction` and `trend_pct`, because the starter-data guidance states that the decline label is derived from them. Also exclude any future-window fields when building a prediction-time feature frame.

**Output:** a ranked review queue that helps an SEO/content editor decide which pages to inspect first.

## 3. Verify it with exactly three warehouse queries

The three checks below verify: **grain**, **March row count/date span**, and **availability using `IS TRUE`**. They intentionally use the March 2026 partition, not `_sample`.

In [ ]:
# Setup: authenticate with the Colab Secret HF_TOKEN; never paste a token into this notebook.
# The exact warehouse query mechanism may be provided by the course environment.
# Replace WAREHOUSE_TABLE below with the course-provided fully-qualified March-partition table reference.
WAREHOUSE_TABLE = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
MONTH = '2026-03'


In [ ]:
# QUERY 1 — Grain: no duplicate client × content × report_date observations in March 2026.
# Expected honest check: zero duplicate groups.
query_1 = f'''
SELECT client_id, content_id, report_date, COUNT(*) AS n
FROM `{WAREHOUSE_TABLE}`
WHERE month = '{MONTH}'
GROUP BY client_id, content_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
'''
print(query_1)
# Execute with the warehouse client supplied by the course environment.

In [ ]:
# QUERY 2 — March row count and date span.
query_2 = f'''
SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM `{WAREHOUSE_TABLE}`
WHERE month = '{MONTH}'
'''
print(query_2)
# Execute with the warehouse client supplied by the course environment.

In [ ]:
# QUERY 3 — Availability: only rows explicitly marked as available survive.
query_3 = f'''
SELECT COUNT(*) AS available_rows
FROM `{WAREHOUSE_TABLE}`
WHERE month = '{MONTH}'
  AND ga4_data_available IS TRUE
'''
print(query_3)
# Execute with the warehouse client supplied by the course environment.

## 3b. Five-feature frame

The five candidate features below are deliberately limited. Each is intended to be knowable at the decision moment. The exact field names should be checked against the warehouse schema before execution.

In [ ]:
feature_frame = [
    ('impressions_prev30', 'Knowable at the decision moment because it summarizes search impressions from the preceding 30-day window.'),
    ('clicks_prev30', 'Knowable at the decision moment because it summarizes clicks from the preceding 30-day window.'),
    ('ctr_prev30', 'Knowable at the decision moment because it is computed from the preceding 30-day search window.'),
    ('gsc_avg_position_prev30', 'Knowable at the decision moment because it summarizes search position from the preceding 30-day window.'),
    ('ga4_data_available', 'Knowable at the decision moment because the warehouse records whether GA4 data is available for the observation.')
]
import pandas as pd
pd.DataFrame(feature_frame, columns=['feature', 'available_when']).

## 3c. The trap — deliberate leakage experiment

For the leakage demonstration, I would deliberately add the outcome-derived field used to define the decline label (for example `trend_direction`) to a quick model/rule. Because the field is derived from the outcome definition, performance should jump toward perfect separation.

**Leakage lesson:** the impressive score is invalid because the feature contains information that would not be safely available at prediction time. The leaked field is then deleted from the feature frame and the honest score is retained.

I will not report a fabricated numeric score here; the score must come from the actual March warehouse run.

## 4. Data limits

**Named limitation:** history depth differs substantially across clients, and some clients have little or no usable search/analytics history. Therefore a single global calendar window can create uneven feature availability and selection bias. The warehouse guidance recommends checking each client's data-start fields and using availability flags rather than treating zero-filled values as genuine zero engagement.

Another important limit is that the June 2026 `_sample` is the final month and should remain sealed while developing future-looking label logic.

## Self-check

- [ ] Five plain-words contract answers are stated.
- [ ] Exactly three verification queries are present.
- [ ] Availability uses `IS TRUE`.
- [ ] Development month is March 2026, not `_sample`.
- [ ] Five features have an `available when?` explanation.
- [ ] Leakage is deliberately demonstrated, then removed.
- [ ] No token, client name, private URL, or private query is committed.
- [ ] The notebook must be executed top-to-bottom in Colab before submission.